In [1]:
%load_ext autoreload
%autoreload 2

import sys
import os
import shutil

import numpy as np
import pandas as pd
import tqdm
import xarray as xr

# Functions for downloading and transforming data
import opensense_data_downloader_and_transformer as oddt

In [2]:
# Folder for downloading data
local_path = "./andersson_2022_OpenMRG/"

# Create folder for storing updated files
os.makedirs("./OpenMRG", exist_ok=True)


# Get original OpenMRG data
source: https://zenodo.org/record/6673751

In [3]:
oddt.download_andersson_2022_OpenMRG(local_path=local_path, print_output=True)

File already exists at desired location ./andersson_2022_OpenMRG/OpenMRG.zip
Not downloading!


# Transform to opensense naming conventions

## CML data

In [4]:
# Transform first part of the data
ds1 = oddt.transform_andersson_2022_OpenMRG(
    fn=local_path + "OpenMRG.zip",  # navigate to your local sandbox clone
    path_to_extract_to=local_path,
    time_start_end=(
        None,
        "2015-07-15T00:00",
    ),  # default (None, None) -> no timeslicing. ie. ('2015-08-31T00', None),
    restructure_data=True,
)

/home/erlend/git/opensense_example_data/OpenMRG/notebooks/opensense_data_downloader_and_transformer.py:303: FutureWarning: the `pandas.MultiIndex` object(s) passed as 'sublink' coordinate(s) or data variable(s) will no longer be implicitly promoted and wrapped into multiple indexed coordinates in the future (i.e., one coordinate for each multi-index level + one dimension coordinate). If you want to keep this behavior, you need to first wrap it explicitly using `mindex_coords = xarray.Coordinates.from_pandas_multiindex(mindex_obj, 'dim')` and pass it as coordinates, e.g., `xarray.Dataset(coords=mindex_coords)`, `dataset.assign_coords(mindex_coords)` or `dataarray.assign_coords(mindex_coords)`.
  ds_multindex = ds.assign_coords({'sublink':df_metadata.index})


In [5]:
# Transform second part of the data
ds2 = oddt.transform_andersson_2022_OpenMRG(
    fn=local_path + "OpenMRG.zip",  # navigate to your local sandbox clone
    path_to_extract_to=local_path,
    time_start_end=(
        "2015-07-15T00:00",
        None,
    ),  # default (None, None) -> no timeslicing. ie. ('2015-08-31T00', None),
    restructure_data=True,
)

/home/erlend/git/opensense_example_data/OpenMRG/notebooks/opensense_data_downloader_and_transformer.py:303: FutureWarning: the `pandas.MultiIndex` object(s) passed as 'sublink' coordinate(s) or data variable(s) will no longer be implicitly promoted and wrapped into multiple indexed coordinates in the future (i.e., one coordinate for each multi-index level + one dimension coordinate). If you want to keep this behavior, you need to first wrap it explicitly using `mindex_coords = xarray.Coordinates.from_pandas_multiindex(mindex_obj, 'dim')` and pass it as coordinates, e.g., `xarray.Dataset(coords=mindex_coords)`, `dataset.assign_coords(mindex_coords)` or `dataarray.assign_coords(mindex_coords)`.
  ds_multindex = ds.assign_coords({'sublink':df_metadata.index})


In [6]:
# concat and drop overlaying duplicate
ds_cml = xr.concat([ds1, ds2], dim="time").drop_duplicates(dim="time")

In [7]:
ds_cml.attrs["file author(s)"] = "Maximilian Graf, Erlend Øydvin and Christian Chwala"
ds_cml.attrs["title"] = "Transformed and resampled OpenMRG-CML"
ds_cml.attrs["comment"] += (
    "\n\nTransformed using opensense_data_downloader_and_transformer \n"
)
ds_cml.attrs["contact"] += ", erlend.oydvin@nmbu.no"

In [8]:
# Create cml folder
os.makedirs("./OpenMRG/cml", exist_ok=True)

# Store transformed CML data
ds_cml.to_netcdf("./OpenMRG/cml/cml.nc")

In [9]:
ds_cml

<xarray.Dataset> Size: 5GB
Dimensions:       (time: 794887, sublink_id: 2, cml_id: 364)
Coordinates:
  * time          (time) datetime64[ns] 6MB 2015-05-31T23:59:00 ... 2015-09-01
  * sublink_id    (sublink_id) <U9 72B 'sublink_1' 'sublink_2'
  * cml_id        (cml_id) int64 3kB 10001 10002 10003 ... 10362 10363 10364
    site_0_lat    (cml_id) float64 3kB 57.7 57.73 57.69 ... 57.65 57.66 57.71
    site_0_lon    (cml_id) float64 3kB 12.0 11.98 11.97 ... 12.12 12.03 12.01
    site_1_lat    (cml_id) float64 3kB 57.7 57.72 57.69 ... 57.66 57.63 57.71
    site_1_lon    (cml_id) float64 3kB 11.99 11.97 11.98 ... 12.14 11.97 11.98
    frequency     (sublink_id, cml_id) float64 6kB 2.821e+04 ... 2.926e+04
    polarization  (sublink_id, cml_id) <U1 3kB 'v' 'v' 'v' 'v' ... 'v' 'v' 'v'
    length        (cml_id) float64 3kB 691.4 614.6 323.7 ... 4.806e+03 1.412e+03
Data variables:
    tsl           (time, sublink_id, cml_id) float32 2GB 1.0 0.0 ... 16.0 0.0
    rsl           (time, sublink_id, cml_id) float32 2GB -49.8 -45.4 ... -52.0
Attributes: (12/14)
    title:                 Transformed and resampled OpenMRG-CML
    version:               1.1
    source:                Swedish Meteorological and Hydrological Institute ...
    contact:               hydro.fou@smhi.se, jafet.andersson@smhi.se, erlend...
    license:               https://creativecommons.org/licenses/by-sa/4.0
    doi:                   https://doi.org/10.5281/zenodo.6673750
    ...                    ...
    institution:           NA
    date:                  NA
    history:               NA
    naming convention:     NA
    license restrictions:  NA
    reference:             NA

## Radar data

In [10]:
# Read radar data and apply opensense naming conventions
ds_rad = (
    xr.open_dataset(local_path + "radar/radar.nc")
    .transpose("time", "y", "x")
)

# Move variables to coordinates
ds_rad.coords['lat'] = ds_rad.lat
ds_rad.coords['lon'] = ds_rad.lon
ds_rad.coords['crs'] = ds_rad.crs

In [11]:
# Make radar directory
os.makedirs("./OpenMRG/radar", exist_ok=True)

# Store transformed radar data
ds_rad.to_netcdf("./OpenMRG/radar/radar.nc")

In [12]:
ds_rad

<xarray.Dataset> Size: 377MB
Dimensions:  (time: 26496, y: 48, x: 37)
Coordinates:
  * time     (time) datetime64[ns] 212kB 2015-06-01T00:05:00 ... 2015-09-01
  * y        (y) float64 384B -3.413e+06 -3.415e+06 ... -3.505e+06 -3.507e+06
  * x        (x) float64 296B -1.542e+05 -1.522e+05 ... -8.42e+04 -8.22e+04
    crs      int32 4B ...
    lat      (y, x) float32 7kB ...
    lon      (y, x) float32 7kB ...
Data variables:
    data     (time, y, x) float64 376MB ...
Attributes:
    source:       Swedish Meteorological and Hydrological Institute (SMHI), H...
    contact:      hydro.fou@smhi.se, remco.vandebeek@smhi.se
    title:        OpenMRG-Radar
    license:      https://creativecommons.org/licenses/by-sa/4.0
    version:      1.1
    doi:          https://doi.org/10.5281/zenodo.6673750
    proj_string:  +proj=stere +lat_ts=60 +ellps=bessel +lon_0=14 +lat_0=90
    comment:      Created by Remco van de Beek, Victor Näslund and Johan Thur...

## Gauge data

In [13]:
# Read city gauge data from CSV, apply opensense naming conventions and store to xarray
df_gauge = pd.read_csv(
    local_path + "gauges/city/CityGauges-2015JJA.csv", index_col=0, parse_dates=True
)
df_gauge_meta = pd.read_csv(local_path + "gauges/city/CityGauges-metadata.csv")

df_gauge.index = df_gauge.index.tz_localize(None).astype("datetime64[ns]")

ds_gauges = xr.Dataset(
    data_vars={"rainfall_amount": (["time", "id"], df_gauge)},
    coords={
        "id": df_gauge_meta.Name.to_numpy(),
        "time": df_gauge.index.to_numpy(),
        "lon": (["id"], df_gauge_meta.Longitude_DecDeg),
        "lat": (["id"], df_gauge_meta.Latitude_DecDeg),
        "location": (["id"], df_gauge_meta.Location),
        "type": (["id"], df_gauge_meta.Type),
        "quantization": (["id"], df_gauge_meta["Resolution (mm)"]),
    },
)

In [14]:
ds_gauges

<xarray.Dataset> Size: 12MB
Dimensions:          (time: 132480, id: 10)
Coordinates:
  * time             (time) datetime64[ns] 1MB 2015-06-01T00:01:00 ... 2015-0...
  * id               (id) object 80B 'Jarn' 'Torp' 'Bergsj' ... 'Lbom' 'Askim'
    lon              (id) float64 80B 11.94 12.04 12.07 ... 11.99 11.97 11.94
    lat              (id) float64 80B 57.65 57.72 57.75 ... 57.71 57.71 57.63
    location         (id) object 80B 'Järnbrottsmotet' ... 'Askim Ögärdesv'
    type             (id) object 80B 'Weighing' 'Weighing' ... 'Tipping-bucket'
    quantization     (id) float64 80B 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.2 0.2 0.2
Data variables:
    rainfall_amount  (time, id) float64 11MB 0.1 0.0 0.2 0.0 ... 0.0 0.0 0.2 0.0

In [15]:
# Read smhio gauge data from CSV, apply opensense naming conventions and store to xarray
df_gauge_smhi = pd.read_csv(
    local_path + "gauges/smhi/GbgA-71420-2015JJA.csv",
    index_col=0,
    parse_dates=True,
)

# Convert to no timezone to make to_numpy work instead of .values (RUFF complains)
df_gauge_smhi.index = df_gauge_smhi.index.tz_localize(None).astype("datetime64[ns]")


ds_gauges_smhi = xr.Dataset(
    data_vars={
        "rainfall_amount": (["time", "id"], df_gauge_smhi.Pvol_mm.to_numpy().reshape(-1, 1)),
    },
    coords={
        "id": ["SMHI"],
        "time": df_gauge_smhi.index.to_numpy(),
        "lon": (["id"], [11.9924]),
        "lat": (["id"], [57.7156]),
        "location": (["id"], ["Goeteburg A"]),
        "type": (["id"], ["15 min rainfall sum"]),
        "quantization": (["id"], [0.1]),
    },
)

In [16]:
ds_gauges_smhi

<xarray.Dataset> Size: 141kB
Dimensions:          (time: 8832, id: 1)
Coordinates:
  * time             (time) datetime64[ns] 71kB 2015-06-01T00:15:00 ... 2015-...
  * id               (id) <U4 16B 'SMHI'
    lon              (id) float64 8B 11.99
    lat              (id) float64 8B 57.72
    location         (id) <U11 44B 'Goeteburg A'
    type             (id) <U19 76B '15 min rainfall sum'
    quantization     (id) float64 8B 0.1
Data variables:
    rainfall_amount  (time, id) float64 71kB 1.0 0.6 0.5 0.6 ... 0.5 0.4 0.5 0.6

In [17]:
# Make gauge directories
os.makedirs("./OpenMRG/gauges", exist_ok=True)
os.makedirs("./OpenMRG/gauges/smhi", exist_ok=True)
os.makedirs("./OpenMRG/gauges/city", exist_ok=True)

# Store data
ds_gauges_smhi.to_netcdf("./OpenMRG/gauges/smhi/smhi_gauge.nc")
ds_gauges.to_netcdf("./OpenMRG/gauges/city/municp_gauge.nc")